# Topic 6: Vision-Language Models (VLM)

**Prerequisites — run these in your terminal before opening this notebook:**
```
brew install ollama
ollama serve &
ollama pull llava
pip install ollama langgraph langchain langchain-core langchain-community ipywidgets opencv-python pillow
```

- **Exercise 1**: Multi-turn LangGraph chat agent that answers questions about an uploaded image
- **Exercise 2**: Video surveillance agent that detects when a person enters/exits a scene

## Exercise 1: Vision-Language LangGraph Chat Agent

In [1]:
import base64
import io
import operator
import ollama
import ipywidgets as widgets
from IPython.display import display, clear_output
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated, List
from PIL import Image

In [2]:
# ── State definition ──────────────────────────────────────────────────────────

class AgentState(TypedDict):
    messages: Annotated[List[dict], operator.add]
    image_b64: str
    response: str


# ── Node ──────────────────────────────────────────────────────────────────────

def call_llava(state: AgentState) -> AgentState:
    """Send the full conversation + image to LLaVA and return the reply."""
    history = state["messages"]
    image_b64 = state["image_b64"]

    # Attach image only to the first user message
    ollama_messages = []
    first_user_seen = False

    for msg in history:
        if msg["role"] == "user" and not first_user_seen:
            first_user_seen = True
            ollama_messages.append({
                "role": "user",
                "content": msg["content"],
                "images": [image_b64]
            })
        else:
            ollama_messages.append({"role": msg["role"], "content": msg["content"]})

    response = ollama.chat(model="llava", messages=ollama_messages)
    reply = response["message"]["content"]

    return {
        "messages": [{"role": "assistant", "content": reply}],
        "image_b64": image_b64,
        "response": reply
    }


# ── Build graph ───────────────────────────────────────────────────────────────

def build_vlm_graph():
    graph = StateGraph(AgentState)
    graph.add_node("llava", call_llava)
    graph.set_entry_point("llava")
    graph.add_edge("llava", END)
    return graph.compile()

vlm_graph = build_vlm_graph()
print("LangGraph VLM agent compiled.")

LangGraph VLM agent compiled.


In [ ]:
# ── UI ────────────────────────────────────────────────────────────────────────

_chat_state: AgentState = {"messages": [], "image_b64": "", "response": ""}

image_output = widgets.Output()
chat_output  = widgets.Output(layout=widgets.Layout(border="1px solid #ccc",
                                                     height="350px",
                                                     overflow_y="auto",
                                                     padding="8px"))
user_input   = widgets.Text(placeholder="Ask something about the image…",
                             layout=widgets.Layout(width="70%"))
send_btn     = widgets.Button(description="Send", button_style="primary")
clear_btn    = widgets.Button(description="Clear Chat", button_style="warning")
upload_btn   = widgets.FileUpload(accept="image/*", multiple=False, description="Upload Image")
status_lbl   = widgets.Label("Upload an image to start.")


def on_upload(change):
    global _chat_state
    if not upload_btn.value:
        return
    # Compatible with both old and new ipywidgets API
    val = upload_btn.value
    if isinstance(val, dict):
        raw = next(iter(val.values()))["content"]
        fname = next(iter(val.keys()))
    else:
        raw = val[0]["content"]
        fname = val[0]["name"]

    # Resize to 512px to keep memory usage reasonable
    img = Image.open(io.BytesIO(bytes(raw)))
    img.thumbnail((512, 512))
    buf = io.BytesIO()
    img.save(buf, format="JPEG")
    img_b64 = base64.b64encode(buf.getvalue()).decode("utf-8")

    _chat_state = {"messages": [], "image_b64": img_b64, "response": ""}

    with image_output:
        clear_output()
        display(img)

    with chat_output:
        clear_output()
    status_lbl.value = f"Loaded: {fname}. Ask a question!"


def on_send(b):
    global _chat_state
    question = user_input.value.strip()
    if not question:
        return
    if not _chat_state["image_b64"]:
        status_lbl.value = "Please upload an image first."
        return

    user_input.value = ""
    status_lbl.value = "Thinking…"
    _chat_state["messages"].append({"role": "user", "content": question})

    try:
        result = vlm_graph.invoke(_chat_state)
        _chat_state["messages"] = result["messages"]
        _chat_state["image_b64"] = result["image_b64"]

        with chat_output:
            print(f"🧑 You: {question}")
            print(f"🤖 LLaVA: {result['response']}")
            print("-" * 60)

        status_lbl.value = "Ready."
    except Exception as e:
        status_lbl.value = f"Error: {e}"


def on_clear(b):
    global _chat_state
    _chat_state["messages"] = []
    with chat_output:
        clear_output()
    status_lbl.value = "Chat cleared."


upload_btn.observe(on_upload, names="value")
send_btn.on_click(on_send)
clear_btn.on_click(on_clear)
user_input.on_submit(on_send)

display(
    widgets.VBox([
        widgets.HTML("<h3>Exercise 1: VLM Chat Agent (LangGraph + LLaVA)</h3>"),
        upload_btn,
        image_output,
        chat_output,
        widgets.HBox([user_input, send_btn, clear_btn]),
        status_lbl
    ])
)

/var/folders/79/vsr3jvfx7lb4nb__4zn51lnw0000gn/T/ipykernel_17579/3741843091.py:88: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  user_input.on_submit(on_send)


---
## Exercise 2: Video Surveillance Agent

In [4]:
import cv2
import base64
import ollama
import ipywidgets as widgets
from IPython.display import display, clear_output

In [5]:
# ── Frame extraction ──────────────────────────────────────────────────────────

def extract_frames(video_path: str, interval_seconds: float = 2.0) -> list:
    """
    Extract one frame every `interval_seconds` from a video.
    Returns a list of (timestamp_seconds, base64_jpeg_string) tuples.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    interval = max(1, int(fps * interval_seconds))
    frames = []
    frame_num = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if frame_num % interval == 0:
            timestamp = frame_num / fps
            _, buf = cv2.imencode(".jpg", frame)
            b64 = base64.b64encode(buf).decode("utf-8")
            frames.append((timestamp, b64))
        frame_num += 1

    cap.release()
    print(f"Extracted {len(frames)} frames from '{video_path}'")
    return frames


# ── Person detector ───────────────────────────────────────────────────────────

DETECT_PROMPT = (
    "Look carefully at this image. "
    "Is there a person (human) visible in the scene? "
    "Answer with only YES or NO."
)

def person_in_frame(image_b64: str) -> bool:
    response = ollama.chat(
        model="llava",
        messages=[{"role": "user", "content": DETECT_PROMPT, "images": [image_b64]}]
    )
    return response["message"]["content"].strip().upper().startswith("YES")


# ── Event detection ───────────────────────────────────────────────────────────

def detect_person_events(frames: list, log_fn=print) -> list:
    events = []
    person_present = False

    for i, (ts, b64) in enumerate(frames):
        log_fn(f"  Checking frame {i+1}/{len(frames)} at {ts:.1f}s …")
        detected = person_in_frame(b64)

        if detected and not person_present:
            events.append({"event": "ENTER", "time": ts})
            log_fn(f"  ✅ Person ENTERED at {ts:.1f}s")
            person_present = True
        elif not detected and person_present:
            events.append({"event": "EXIT", "time": ts})
            log_fn(f"  🚪 Person EXITED at {ts:.1f}s")
            person_present = False

    return events


def format_time(seconds: float) -> str:
    m, s = divmod(int(seconds), 60)
    return f"{m:02d}:{s:02d}"


print("Surveillance helpers loaded.")

Surveillance helpers loaded.


In [ ]:
# ── UI ────────────────────────────────────────────────────────────────────────

interval_slider  = widgets.FloatSlider(value=2.0, min=0.5, max=10.0, step=0.5,
                                       description="Interval (s):",
                                       style={"description_width": "initial"})
upload_video_btn = widgets.FileUpload(accept="video/*", multiple=False, description="Upload Video")
run_btn          = widgets.Button(description="Run Surveillance", button_style="danger",
                                  disabled=True, layout=widgets.Layout(width="200px"))
surv_output      = widgets.Output(layout=widgets.Layout(border="1px solid #ccc",
                                                         height="400px",
                                                         overflow_y="auto",
                                                         padding="8px"))
surv_status      = widgets.Label("Upload a video to begin.")

_video_path = None


def on_upload_video(change):
    global _video_path
    if not upload_video_btn.value:
        return
    val = upload_video_btn.value
    if isinstance(val, dict):
        raw = next(iter(val.values()))["content"]
        fname = next(iter(val.keys()))
    else:
        raw = val[0]["content"]
        fname = val[0]["name"]

    save_path = f"/tmp/{fname}"
    with open(save_path, "wb") as f:
        f.write(bytes(raw))
    _video_path = save_path
    run_btn.disabled = False
    surv_status.value = f"Loaded: {fname}. Press Run."


def on_run(b):
    if not _video_path:
        surv_status.value = "Please upload a video first."
        return

    run_btn.disabled = True
    surv_status.value = "Running… (this may take several minutes)"

    with surv_output:
        clear_output()
        print(f"Video: {_video_path}")
        print(f"Frame interval: {interval_slider.value}s\n")
        try:
            frames = extract_frames(_video_path, interval_seconds=interval_slider.value)
            events = detect_person_events(frames, log_fn=print)

            print("\n" + "=" * 50)
            print("SURVEILLANCE REPORT")
            print("=" * 50)
            if not events:
                print("No person detected in the video.")
            else:
                for ev in events:
                    icon = "➡️ " if ev["event"] == "ENTER" else "⬅️ "
                    print(f"{icon} Person {ev['event']} at {format_time(ev['time'])} ({ev['time']:.1f}s)")
            print("=" * 50)
        except Exception as e:
            print(f"ERROR: {e}")

    run_btn.disabled = False
    surv_status.value = "Done."


upload_video_btn.observe(on_upload_video, names="value")
run_btn.on_click(on_run)

display(
    widgets.VBox([
        widgets.HTML("<h3>Exercise 2: Video Surveillance Agent</h3>"),
        upload_video_btn,
        interval_slider,
        run_btn,
        surv_output,
        surv_status
    ])
)